# Musanga V4 — NB05 v4_001 | RGB vs multiscale DSM vs multispectral (matched evaluation)

## Scientific comparison — NB04 equal-footprint reporting

**Frozen Model A:** use the **actual NB03 RGB Model A** (not earlier NB02/V3.1 weights). **Held-out references:** same 30 circles + 76 plots, spatial groups and outer folds as **NB04 v4_006**, including the four difficult MOS references. No relabeling, no post-hoc trimming.

| Arm | Model | Input | Training |
|---|---|---|---|
| RGB | Original Model A | RGB (3 bands) | **Frozen**, previously trained in NB03 |
| RGB+DSM | ResNet18 U-Net | RGB + 10 DSM features | Initialized from original Model A; fine-tune on the **same baseline-only labelled training chips** used by the MS arm |
| RGB+MS | ResNet18 U-Net | RGB + 4 relative MicaSense bands (R,G,red-edge,NIR) | Same training protocol, support and schedule as DSM |

DSM comprises **3 local features** (prominence, roughness, convexity) plus **7 multiscale context features**: relative elevation in 5–15, 15–30 and 30–60 m neighborhoods; surface SD at 15, 30 and 60 m; and valid fraction at 30–60 m. Derived from earlier NB03 DSM v4_009 / historical NB05 v4_026, including halo-safe windows.

**Critical comparison rule:** The full NB04 RGB summary for 106 units remains the *external baseline*. The direct 3-arm test evaluates **all models on identical pixels** for which RGB+DSM+all four MS bands are valid, with **no spectral imputation or cross-flight band mixing**. Some of the 106 units may be only partially supported or have no complete MS coverage: do not claim 106 complete 3-way units until verified.

**Interpretation:** the DSM and MS arms are *initialized from* the frozen RGB model but are fine-tuned; baseline is not. Thus this tests incremental fine-tuning with auxiliary sensor features, **not** an exactly controlled RGB-only retraining ablation. A separately trained RGB-on-common-support arm would be required to isolate modality from additional optimization.

Run cells in order on Colab GPU: **00–02 preflight → 03 cache → 04 train → 05 matched inference → 06 report**. Every substantial operation is resume-safe; **DO NOT rerun NB03 or NB04**. The historical notebook had very slow DSM context calculation; this implementation maintains the validated exact feature recipe first, caching training chips and per-unit outputs, not applying numerical approximations without measurement. No completed results are claimed before Colab execution.

**Important:** Multispectral alignment is only as good as source co-registration. Inspect registration and valid coverage before interpreting a potential modality improvement.


## 00 — Immutable NB04 references, folds, Model A checkpoints and sources

In [ ]:
%pip -q install geopandas pyogrio shapely rasterio scipy pandas scikit-learn matplotlib segmentation-models-pytorch
from pathlib import Path
import json, hashlib, gc, warnings, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import transform as shape_transform
from pyproj import Transformer
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from scipy.special import expit
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                        Path('/content/drive/MyDrive/Musanga/Final')] if p.exists()),None)
assert ROOT is not None, 'Musanga Final folder not found'
OLD=ROOT/'Outputs/V3_1'; AUDIT=ROOT/'Outputs/v4/00_input_audit'
SPLIT_ROOT=ROOT/'Outputs/v4/01_spatial_cover_folds_v2'
NB02=ROOT/'Outputs/v4/02_rgb_circles_plus_plots_v3'
NB03=ROOT/'Outputs/v4/03_paired_rgb_unet_v1'
OUT=ROOT/'Outputs/v4/05_rgb_dsm_ms_matched_v1';OUT.mkdir(parents=True,exist_ok=True)
NB04_OUT=ROOT/'Outputs/v4/04_paired_rgb_heldout_v1'
V31=json.loads((OLD/'00_input_audit/00_input_manifest.json').read_text())
F78=json.loads((AUDIT/'00_v4_field78_contract.json').read_text())
SPLIT_CONTRACT=json.loads((SPLIT_ROOT/'01_reference_fold_contract_v2.json').read_text())
TRAIN=json.loads((NB03/'03_training_contract.json').read_text())
FOLD_CSV=Path(SPLIT_CONTRACT['output_csv'])
assert hashlib.sha256(FOLD_CSV.read_bytes()).hexdigest()==SPLIT_CONTRACT['csv_sha256']
assert TRAIN['frozen_nb01_hash']==SPLIT_CONTRACT['csv_sha256']
SPLITS=pd.read_csv(FOLD_CSV)
assert len(SPLITS)==108 and (SPLITS.unit_type=='circle').sum()==30 and (SPLITS.unit_type=='plot').sum()==78
UNITS=gpd.read_file(SPLIT_ROOT/'01_reference_groups_and_folds_v2.gpkg').to_crs('EPSG:32635')
assert len(UNITS)==108 and not UNITS.duplicated(['unit_type','unit_id']).any()
OWNERS=pd.read_csv(NB02/'02_reference_mosaic_owners_v2.csv')
assert not OWNERS.duplicated(['unit_type','unit_id']).any()
EXCLUDED={('plot','Mixt1'),('plot','Mixt2')}
assert set(map(tuple,OWNERS.loc[OWNERS.roi_mosaic_coverage_fraction.lt(.95),['unit_type','unit_id']].astype(str).values))==EXCLUDED
EVAL=UNITS.merge(OWNERS[['unit_type','unit_id','mosaic','roi_mosaic_coverage_fraction']],
    on=['unit_type','unit_id'],validate='one_to_one')
EVAL=EVAL[EVAL.roi_mosaic_coverage_fraction.ge(.95)].copy()
assert len(EVAL)==106 and (EVAL.unit_type=='circle').sum()==30 and (EVAL.unit_type=='plot').sum()==76
assert set(TRAIN['checkpoints'])=={str(NB03/f'fold{f}_{a}_{TRAIN["training_key"][:18]}.pt') for f in range(3) for a in 'AB'}
assert all(Path(p).is_file() for p in TRAIN['checkpoints'])
MOSAICS={}
for q in V31['mosaics']:
    if q['file'].endswith('.tif'):
        p=Path(q['path'])
        if not p.exists(): p=ROOT/'Data/Ortho_RGB'/q['file']
        assert p.exists(),p
        MOSAICS[q['file']]=p
assert len(MOSAICS)==6
print('HELD-OUT UNITS:',EVAL.groupby(['fold','site','unit_type']).size().to_string())
print('CHECKPOINTS:',len(TRAIN['checkpoints']),'TRAINING KEY:',TRAIN['training_key'][:18]);print('OUTPUT:',OUT)
assert (NB04_OUT/'04_heldout_unit_metrics.csv').is_file(),'NB04 completed result CSV is missing'
BASELINE=pd.read_csv(NB04_OUT/'04_heldout_unit_metrics.csv').query('arm == "A"').copy()
assert len(BASELINE)==106
CHIP_QA=pd.read_csv(NB03/'03_complete_chip_QA.csv')
assert {'cache_file','cache_index','usable_A','fold','site','mosaic','row','col'}.issubset(CHIP_QA.columns)
print('Available baseline-labelled RGB chips:',int(CHIP_QA.usable_A.sum()),'| NB04 baseline units:',len(BASELINE))


## 01 — Reference geometry and exact NB04 supervision semantics

In [ ]:
from shapely import make_valid
from shapely.ops import unary_union
from shapely.geometry import GeometryCollection
REF_CRS='EPSG:32635'
def repair(g):
    if g is None or g.is_empty: raise ValueError('Empty reference geometry')
    if not g.is_valid:
        z=make_valid(g)
        pieces=[h for h in getattr(z,'geoms',[z]) if h.geom_type in ('Polygon','MultiPolygon')]
        g=unary_union(pieces)
    assert g.is_valid and not g.is_empty
    return g
def union(g):
    arr=list(g)
    return unary_union(arr) if arr else GeometryCollection()
CIRCLE_PATH=Path(F78['circle_complete_roi']); EXH_PATH=Path(F78['circle_positive_crowns'])
PLOT_PATH=Path(F78['plot_file']); CROWN_PATH=Path(F78['exhaustive_plot_crowns'])
# NB03-compatible filtering: the raw circle source can contain additional circles.
# Preserve source index identifiers BEFORE subsetting to the frozen NB01 units.
circle_src=gpd.read_file(CIRCLE_PATH).to_crs(REF_CRS)
circle_src['unit_id']=circle_src.index.astype(str)
plot_src=gpd.read_file(PLOT_PATH).to_crs(REF_CRS)
plot_src['unit_id']=plot_src.Plot_name.astype(str)
expected_circles=set(UNITS.loc[UNITS.unit_type.eq('circle'),'unit_id'].astype(str))
expected_plots=set(UNITS.loc[UNITS.unit_type.eq('plot'),'unit_id'].astype(str))
assert len(expected_circles)==30 and len(expected_plots)==78
assert expected_circles.issubset(set(circle_src.unit_id)), 'Frozen circle IDs missing from source'
assert expected_plots.issubset(set(plot_src.unit_id)), 'Frozen plot IDs missing from source'
print('Raw source reference counts:',len(circle_src),'circles,',len(plot_src),'plots')
circle_src=circle_src[circle_src.unit_id.isin(expected_circles)].copy()
plot_src=plot_src[plot_src.unit_id.isin(expected_plots)].copy()
assert len(circle_src)==30 and len(plot_src)==78, 'Filtered sources disagree with frozen NB01 contract'
# Repair using identical NB03 polygonal semantics, with explicit identifiers.
circle_src.geometry=[repair(g) for g in circle_src.geometry]
plot_src.geometry=[repair(g) for g in plot_src.geometry]
exh=gpd.read_file(EXH_PATH).to_crs(REF_CRS);exh.geometry=[repair(g) for g in exh.geometry]
crowns=gpd.read_file(CROWN_PATH).to_crs(REF_CRS);crowns.geometry=[repair(g) for g in crowns.geometry]
crowns['is_musanga']=pd.to_numeric(crowns.Class_ID,errors='raise').eq(1)
KVO_POS=union(exh.geometry)
NADINE_POS=union(crowns.loc[crowns.is_musanga].geometry)
NADINE_NEG=union(crowns.loc[~crowns.is_musanga].geometry)
ROI=gpd.read_file(Path(V31['roi'])).to_crs(REF_CRS)
ROI_GEOM=union(ROI.geometry)
# Unions are never used to redefine unit identity. Each unit retains its own extent.
REF_UNITS={('circle',str(z.unit_id)):z.geometry for z in circle_src.itertuples()}
REF_UNITS.update({('plot',str(z.unit_id)):z.geometry for z in plot_src.itertuples()})
assert all((str(z.unit_type),str(z.unit_id)) in REF_UNITS for z in EVAL.itertuples())
# Exclude only genuine annotation disagreement on circle/plot shared footprints.
SHARED=union(circle_src.geometry).intersection(union(plot_src.geometry))
CONFLICT=(KVO_POS.intersection(SHARED).symmetric_difference(
    NADINE_POS.intersection(SHARED))).union(NADINE_POS.intersection(NADINE_NEG))
assert CONFLICT.is_valid
print('REFERENCE:',len(REF_UNITS),'| conflict area m²:',round(CONFLICT.area,1))
print('WARNING: plot background = absent Musanga only if annotations are exhaustive, as assumed by NB03.')

## 02 — Historical audited RGB–DSM pairings and 4-band MS flight registry

In [ ]:
import os,re
from collections import defaultdict
from shapely.geometry import box
DATA=ROOT/'Data'
# Historical exact RGB–DSM pairing is read from NB00; do not guess DSM filenames.
N0=AUDIT
REGISTRY_OUT=OUT/'02_sensor_registry'
REGISTRY_OUT.mkdir(exist_ok=True)
PAIRS=pd.read_csv(N0/'00_rgb_dsm_pair_audit.csv')
assert len(PAIRS)==6 and PAIRS.rgb.is_unique
RGB_DIR=DATA/'Ortho_RGB'
REG=pd.read_csv(N0/'00_ms_raster_headers.csv')
REG=REG.loc[REG.error.fillna('').eq('')]
MS_BANDS={}
RGB_TO_SITE={}
for record in PAIRS.itertuples(index=False):
    p=Path(record.rgb_path)
    assert p.is_file() and Path(record.dsm_path).is_file()
    RGB_TO_SITE[p.name]=str(next(iter(EVAL.loc[EVAL.mosaic.eq(p.name),'site'].unique())))
assert len(RGB_TO_SITE)==6
# Preserve exact historical four-band selection; P1/P2 are added separately.
BASE_GROUPS=['Baego_MS','MOS_1_2_MS','MOS_3_4_MS','MOS_5_6_MS',
             'Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS']
MOSAIC_KEY={
'Baego_MS':'Baego.tif',
'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
for group in BASE_GROUPS:
    part=REG[REG.group.eq(group)]
    if group=='Baego_MS':
        assert len(part)==1,(group,len(part))
        p=Path(part.iloc[0].path)
        with rasterio.open(p) as src:
            assert src.count>=4
        MS_BANDS[group]={b:p for b in ('red','green','red_edge','nir')}
    else:
        files={}
        for b in ('red','green','red_edge','nir'):
            matches=part.loc[part.band_hint.eq(b)]
            if b=='red_edge' and group=='MOS_1_2_MS' and len(matches)>1:
                matches=matches.loc[matches.name.eq(
                    'mlt_Mos_1_2_transparent_mosaic_red edge_georef.tif')]
            assert len(matches)==1,(group,b,list(matches.name))
            files[b]=Path(matches.iloc[0].path)
        MS_BANDS[group]=files
# Enumerate only the new Yoko names, with no image pixel reads.
yoko_root=DATA
yoko_folders=[]
for folder,dirs,files in os.walk(yoko_root):
    for d in dirs:
        if 'carlier2020_yoko_micasense' in d.lower():
            yoko_folders.append(Path(folder)/d)
    dirs[:]=[d for d in dirs if d not in
        {'Outputs','.git','__pycache__'}]
# Folders may also be dataset filename prefixes rather than directory names.
raw=[]
for p in yoko_folders:
    raw.extend(q for q in p.rglob('*') if q.is_file() and q.suffix.lower() in ('.tif','.tiff'))
# Support plain TIFF filenames stored directly in any DATA subdirectory.
for folder,dirs,files in os.walk(yoko_root):
    for name in files:
        if ('carlier2020_yoko_micasense' in name.lower() and
            name.lower().endswith(('.tif','.tiff'))):
            raw.append(Path(folder)/name)
    dirs[:]=[d for d in dirs if d not in {'Outputs','.git','__pycache__'}]
raw=sorted(set(raw),key=str)
def classify(name):
    s=re.sub('[^a-z0-9]+','_',name.lower())
    if re.search(r'(^|_)(red_edge|rededge|red_?edge)(_|$)',s):return 'red_edge'
    if re.search(r'(^|_)(nir|near_infrared)(_|$)',s):return 'nir'
    if re.search(r'(^|_)green(_|$)',s):return 'green'
    if re.search(r'(^|_)red(_|$)',s):return 'red'
    return None
YOKO={}
for flight in ('P1','P2'):
    matches=[]
    for p in raw:
        s=str(p).lower()
        if re.search(r'(^|[_/\\-])'+flight.lower()+r'($|[_/\\.-])',s):
            # filename first, then parent band-labeled folder
            band=classify(p.stem) or classify(p.parent.name)
            if band:matches.append((band,p))
    found=defaultdict(list)
    for band,p in matches:found[band].append(p)
    for band in ('red','green','red_edge','nir'):
        if len(found[band])!=1:
            print('YOKO',flight,band,'candidates',list(map(str,found[band])))
    if all(len(found[b])==1 for b in ('red','green','red_edge','nir')):
        group=f'Yoko_Carlier2020_MicaSense_{flight}_MS'
        YOKO[group]={b:found[b][0] for b in ('red','green','red_edge','nir')}
        yoko_rgb=[p for p in RGB_DIR.glob('*.tif') if 'yoko' in p.name.lower()]
        assert len(yoko_rgb)==1,('Yoko RGB mosaic ambiguity',yoko_rgb)
        MOSAIC_KEY[group]=yoko_rgb[0].name
        MS_BANDS[group]=YOKO[group]
    else:
        print('YOKO FLIGHT NOT REGISTERED',flight,'manual filename validation required')
print('DISCOVERED YOKO SPECTRAL FILES:',len(raw))
display(pd.DataFrame([dict(flight=k,band=b,path=str(p)) for k,ff in YOKO.items()
                     for b,p in ff.items()]))
assert len(YOKO)==2,'Need 4 unambiguous first-band TIFFs for each P1 and P2 before training'
HEADERS=[]
for group,files in MS_BANDS.items():
    for band,p in files.items():
        with rasterio.open(p) as src:
            assert src.count>=1 and src.crs is not None,(group,band,p)
            index=('red','green','red_edge','nir').index(band)+1 if group=='Baego_MS' else 1
            assert src.count>=index
            HEADERS.append(dict(group=group,band=band,filename=p.name,
                 data_band=index,path=str(p),crs=str(src.crs),width=src.width,
                 height=src.height,bounds=str(src.bounds),pixel_m=abs(src.transform.a)))
HEADERS=pd.DataFrame(HEADERS)
HEADERS.to_csv(OUT/'05_harmonized_sensor_registry.csv',index=False)
display(HEADERS[['group','band','filename','data_band','pixel_m']])

DSM_BY_MOSAIC={Path(x.rgb_path).name:Path(x.dsm_path) for x in PAIRS.itertuples(index=False)}
GROUPS_BY_MOSAIC=defaultdict(list)
for group,mosaic in MOSAIC_KEY.items():GROUPS_BY_MOSAIC[mosaic].append(group)
assert set(EVAL.mosaic).issubset(DSM_BY_MOSAIC),'Some RGB mosaics missing audited DSM'
assert set(EVAL.mosaic).issubset(GROUPS_BY_MOSAIC),'Some RGB mosaics missing spectral flight registry'
print('Audited RGB–DSM pairs:',len(DSM_BY_MOSAIC),'| MS flights:',len(MS_BANDS))


### Multiscale DSM feature extraction — exact historical features (10 channels)

In [ ]:
PROMINENCE_RADIUS_M=6.
ROUGHNESS_RADIUS_M=3.
CONVEXITY_RADIUS_M=2.
CONTEXT_RESOLUTION_M=1.
CONTEXT_MAX_RADIUS_M=60.
# 02 — Windowed, halo-safe structural feature extraction
from contextlib import ExitStack
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from scipy.ndimage import uniform_filter
def _mean_valid(a,valid,size):
    f=valid.astype(np.float32)
    num=uniform_filter(np.where(valid,a,0.).astype(np.float32),size=size,mode='nearest')
    den=uniform_filter(f,size=size,mode='nearest')
    return num/np.maximum(den,1e-6),den

def _odd_pixels(radius_m,pixel_m):
    r=max(1,int(np.ceil(radius_m/pixel_m)))
    return 2*r+1

def local_feature_tile(rgb_path,dsm_path,win):
    """Return 6xHxW [RGB3, prominence, roughness, convexity], joint valid HxW.
    Requires actual pixel registration to be separately approved before training.
    """
    with ExitStack() as stack:
        # Accept already-open raster handles to avoid hundreds of repeated GDAL opens.
        # Original path-based calls (including held-out inference) still work.
        rgb=(rgb_path if hasattr(rgb_path,'read') else stack.enter_context(rasterio.open(rgb_path)))
        dsm=(dsm_path if hasattr(dsm_path,'read') else stack.enter_context(rasterio.open(dsm_path)))
        assert rgb.crs is not None and dsm.crs is not None
        h,w=int(win.height),int(win.width)
        assert h>0 and w>0 and int(win.row_off)>=0 and int(win.col_off)>=0
        assert win.row_off+h<=rgb.height and win.col_off+w<=rgb.width
        step=min(abs(rgb.transform.a),abs(rgb.transform.e))
        max_radius=max(PROMINENCE_RADIUS_M,ROUGHNESS_RADIUS_M,
                       CONVEXITY_RADIUS_M)
        halo=int(np.ceil(2.0*max_radius/step))+4
        r0=max(0,int(win.row_off)-halo);c0=max(0,int(win.col_off)-halo)
        r1=min(rgb.height,int(win.row_off)+h+halo)
        c1=min(rgb.width,int(win.col_off)+w+halo)
        expanded=Window(c0,r0,c1-c0,r1-r0)
        tr=rgb.window_transform(expanded)
        rgb_raw=rgb.read([1,2,3],window=expanded,masked=True)
        if not all(np.dtype(t)==np.dtype('uint8') for t in rgb.dtypes[:3]):
            raise ValueError(f'RGB is not uint8: {rgb_path}: {rgb.dtypes}')
        rgb_valid=(rgb.dataset_mask(window=expanded)>0)&(
            ~np.ma.getmaskarray(rgb_raw).any(axis=0))
        rgb_data=np.asarray(rgb_raw.filled(0),dtype=np.float32)/255.
        vrt=stack.enter_context(WarpedVRT(dsm,crs=rgb.crs,transform=tr,
            width=int(expanded.width),height=int(expanded.height),
            resampling=Resampling.bilinear,src_nodata=dsm.nodata,
            nodata=np.nan))
        arr=vrt.read(1,masked=True)
        z=np.asarray(arr.data,dtype=np.float32).copy()
        dsm_valid=(~np.ma.getmaskarray(arr))&np.isfinite(z)
        valid=rgb_valid&dsm_valid
        z0=np.where(dsm_valid,z,0.).astype(np.float32)
        prominence_size=_odd_pixels(PROMINENCE_RADIUS_M,step)
        rough_size=_odd_pixels(ROUGHNESS_RADIUS_M,step)
        convex_size=_odd_pixels(CONVEXITY_RADIUS_M,step)
        base,base_support=_mean_valid(z0,dsm_valid,prominence_size)
        local,rough_support=_mean_valid(z0,dsm_valid,rough_size)
        local2,_=_mean_valid(z0*z0,dsm_valid,rough_size)
        smooth,conv_support=_mean_valid(z0,dsm_valid,convex_size)
        prominence=z0-base
        roughness=np.sqrt(np.maximum(0.,local2-local**2))
        convexity=z0-smooth
        # Exclude filters with inadequate valid support near DSM boundaries.
        valid &= (base_support>=.95)&(rough_support>=.95)&(conv_support>=.95)
        y=int(win.row_off)-r0;x=int(win.col_off)-c0
        sl=np.s_[y:y+h,x:x+w]
        x6=np.concatenate([rgb_data[:,sl[0],sl[1]],
            prominence[sl][None],roughness[sl][None],
            convexity[sl][None]],axis=0).astype(np.float32)
        joint=valid[sl].copy()
        x6[:,~joint]=0.
        assert x6.shape==(6,h,w) and joint.shape==(h,w)
        assert np.isfinite(x6).all()
        return x6,joint

# 1 m global UTM-anchored context grid: consistent regardless of chip positions.
from rasterio.warp import reproject
from rasterio.transform import Affine
from scipy.ndimage import uniform_filter
from math import floor,ceil

CONTEXT_BANDS=('relative_surface_5_15m','relative_surface_15_30m',
    'relative_surface_30_60m','surface_sd_15m','surface_sd_30m',
    'surface_sd_60m','context_valid_fraction_30_60m')
FEATURE_NAMES=('R','G','B','local_prominence_m','surface_roughness_m',
               'local_convexity_m')+CONTEXT_BANDS
N_DSM_FEATURES=len(FEATURE_NAMES)-3
assert N_DSM_FEATURES==10  # 3 local + 7 contextual
def _box_mean(data,valid,r):
    size=2*int(r)+1
    mask=valid.astype(np.float32)
    val=np.where(valid,data,0.).astype(np.float32)
    count=uniform_filter(mask,size=size,mode='constant',cval=0.)
    total=uniform_filter(val,size=size,mode='constant',cval=0.)
    return total/np.maximum(count,1e-6),count

def _annular_mean(data,valid,inner,outer):
    # uniform_filter returns average over a square window; reconstruct
    # weighted sums for annular mean, excluding its inner square.
    def sums(radius):
        size=2*radius+1
        cnt=uniform_filter(valid.astype(np.float32),size=size,mode='constant',cval=0.)*(size*size)
        val=uniform_filter(np.where(valid,data,0.).astype(np.float32),size=size,
                           mode='constant',cval=0.)*(size*size)
        return val,cnt
    so,no=sums(int(outer))
    si,ni=sums(int(inner))
    count=np.maximum(no-ni,0.)
    return (so-si)/np.maximum(count,1e-6),np.clip(count/max(1,(2*outer+1)**2-(2*inner+1)**2),0.,1.)

def context_feature_tile(rgb,dsm,win):
    """Seven spatial contextual features at native RGB grid, coarse computation at 1 m."""
    bounds=rasterio.windows.bounds(win,rgb.transform)
    xmin,ymin,xmax,ymax=bounds
    pad=CONTEXT_MAX_RADIUS_M+4*CONTEXT_RESOLUTION_M
    unit=CONTEXT_RESOLUTION_M
    left=floor((xmin-pad)/unit)*unit
    right=ceil((xmax+pad)/unit)*unit
    top=ceil((ymax+pad)/unit)*unit
    bottom=floor((ymin-pad)/unit)*unit
    w=int(round((right-left)/unit));h=int(round((top-bottom)/unit))
    assert 0<w<=8192 and 0<h<=8192,(h,w,'Unexpected context window')
    tr=Affine(unit,0,left,0,-unit,top)
    with WarpedVRT(dsm,crs=rgb.crs,transform=tr,width=w,height=h,
                   resampling=Resampling.bilinear,src_nodata=dsm.nodata,nodata=np.nan) as vrt:
        raster=vrt.read(1,masked=True)
    z=np.asarray(raster.data,dtype=np.float32)
    valid=(~np.ma.getmaskarray(raster))&np.isfinite(z)
    # Preserve relative heights but remove global offset before variance estimation.
    reference=float(np.median(z[valid])) if np.any(valid) else 0.
    relative=np.where(valid,z-reference,0.).astype(np.float32)
    local,_=_box_mean(relative,valid,3)
    surfaces=[];fractions=[]
    for inner,outer in ((5,15),(15,30),(30,60)):
        surround,frac=_annular_mean(relative,valid,inner,outer)
        surfaces.append(np.where(frac>=.6,local-surround,0.).astype(np.float32))
        fractions.append(frac)
    rough=[]
    for radius in (15,30,60):
        mean,cov=_box_mean(relative,valid,radius)
        second,_=_box_mean(relative*relative,valid,radius)
        rough.append(np.where(cov>=.6,np.sqrt(np.maximum(second-mean*mean,0.)),0.).astype(np.float32))
    coarse=np.stack(surfaces+rough+[fractions[2]],axis=0)
    dst_transform=rgb.window_transform(win)
    out=np.zeros((len(CONTEXT_BANDS),int(win.height),int(win.width)),dtype=np.float32)
    # Smooth resampling of contextual maps. No extrapolation beyond coarse coverage.
    for i in range(len(CONTEXT_BANDS)):
        reproject(source=coarse[i],destination=out[i],
                  src_transform=tr,src_crs=rgb.crs,dst_transform=dst_transform,
                  dst_crs=rgb.crs,resampling=Resampling.bilinear,
                  src_nodata=None,dst_nodata=-9999.)
    # 0 is a legitimate relative-elevation/roughness value. It must NOT be
    # used as destination NoData: GDAL may perturb valid zeros to 1.4013e-45.
    # Convert only pixels explicitly initialized to -9999 back to 0 in the
    # final feature tensor (the contextual coverage band carries availability).
    out[out==-9999.]=0.
    assert np.isfinite(out).all()
    return out

def feature_tile(rgb_path,dsm_path,win):
    # Handles may be passed directly by the 25-chip persistent cache.
    with ExitStack() as stack:
        rgb=rgb_path if hasattr(rgb_path,'read') else stack.enter_context(rasterio.open(rgb_path))
        dsm=dsm_path if hasattr(dsm_path,'read') else stack.enter_context(rasterio.open(dsm_path))
        local,joint=local_feature_tile(rgb,dsm,win)
        ctx=context_feature_tile(rgb,dsm,win)
    out=np.concatenate([local,ctx],axis=0)
    out[:,~joint]=0.
    assert out.shape==(3+N_DSM_FEATURES,int(win.height),int(win.width))
    assert np.isfinite(out).all()
    return out,joint

print('DSM channels:',FEATURE_NAMES)


### Four-band same-flight MS resampling (relative spectra, no cross-flight mixing)

In [ ]:
from contextlib import ExitStack
from rasterio.vrt import WarpedVRT
def bands_vrt08(rgb,groups,stack):
    result={}
    for group in groups:
        result[group]={}
        for band in ('red','green','red_edge','nir'):
            src=stack.enter_context(rasterio.open(MS_BANDS[group][band]))
            idx=('red','green','red_edge','nir').index(band)+1 if group=='Baego_MS' else 1
            vrt=stack.enter_context(WarpedVRT(src,crs=rgb.crs,
                transform=rgb.transform,width=rgb.width,height=rgb.height,
                resampling=Resampling.bilinear,dtype='float32'))
            result[group][band]=(vrt,idx)
    return result
def matched_ms08(vrts,win):
    stack=[];supported=[]
    for band in ('red','green','red_edge','nir'):
        ds,idx=vrts[band]
        a=ds.read(idx,window=win,masked=True,out_dtype='float32')
        v=np.asarray(a.filled(0),np.float32)
        ok=(~np.ma.getmaskarray(a))&np.isfinite(v)&(v>0)
        stack.append(np.where(ok,v,0))
        supported.append(ok)
    raw=np.stack(stack)
    valid=np.logical_and.reduce(supported)
    total=raw.sum(axis=0)
    valid &=total>1e-6
    relative=np.where(valid[None],raw/np.maximum(total[None],1e-6),0)
    return relative.astype('float16'),valid

def mosaic_ms_vrts(rgb,stack,mosaic):
    groups=GROUPS_BY_MOSAIC.get(mosaic,[])
    assert groups,('No four-band source for RGB mosaic',mosaic)
    return bands_vrt08(rgb,groups,stack)

def choose_complete_flight(vrts,win):
    # One acquisition supplies ALL FOUR spectral bands for each tile.
    # Yoko P1/P2 are compared as separate complete acquisitions, never mixed.
    candidates=[(group,*matched_ms08(channels,win))
                for group,channels in vrts.items()]
    group,ms,mask=max(candidates,key=lambda z:int(z[2].sum()))
    return ms.astype('float32'),mask,group


## 03 — Resume-safe common-support labelled training cache (Model A labels ONLY)

Only NB03 `usable_A` chips (exhaustive circles and supplementary partial crown annotations) are used. No B plot-label additions. A deterministic per-site cap limits one-time DSM extraction; **both fusion arms use exactly the same chips, pixels and labels**. Fixed spatially-blocked inner validation is derived from chip coordinates, never from held-out reference scores.

In [ ]:
import hashlib,os,gc,time
from rasterio.windows import Window
from collections import defaultdict
CACHE=OUT/'03_common_support_chips'
CACHE.mkdir(exist_ok=True)
MAX_TRAIN_CHIPS_PER_SITE_FOLD=160   # efficient pre-registered cap, identical for DSM and MS
MIN_SUPERVISED_PIXELS=250
TRAIN_SEED=20261119
CAND=CHIP_QA[CHIP_QA.usable_A.eq(True)].copy()
# Reproducibly mix circle and partial sources; do not truncate an alphabetically
# sorted list, which would disproportionately remove non-exhaustive crown chips.
CAND['sample_priority']=[
    int(hashlib.sha256(repr((TRAIN_SEED,int(q.fold),q.site,q.mosaic,
        q.source,int(q.row),int(q.col))).encode()).hexdigest()[:12],16)
    for q in CAND.itertuples(index=False)]
CAND=CAND.sort_values(['fold','site','sample_priority'],kind='stable')
CAND=CAND.groupby(['fold','site'],group_keys=False).head(MAX_TRAIN_CHIPS_PER_SITE_FOLD).copy()
print('Selected RGB Model A chip sources (before sensor validity filtering):')
display(CAND.groupby(['fold','site','source']).size().rename('n_chips').reset_index())
assert CAND.fold.nunique()==3 and len(CAND)>100
# Training data manifest: original RGB cache key and file signatures + selected windows.
def sig(files):
    return [(str(p),int(Path(p).stat().st_size),int(Path(p).stat().st_mtime_ns))
            for p in sorted(set(map(Path,files)),key=str)]
def inner_role(z,ds):
    # Spatial block IDs in metres; no tile within a 50m block can cross train/val.
    xx,yy=ds.xy(int(z.row)+128,int(z.col)+128)
    key=f'{z.site}_{int(np.floor(xx/50))}_{int(np.floor(yy/50))}'
    seed=int(hashlib.sha256(key.encode()).hexdigest()[:12],16)
    return 'val' if seed%5==0 else 'train'

CACHE_ROWS=[]
for (fold,mosaic),g in CAND.groupby(['fold','mosaic'],sort=True):
    refs=[MOSAICS[mosaic],DSM_BY_MOSAIC[mosaic]]
    for name in GROUPS_BY_MOSAIC[mosaic]:refs+=list(MS_BANDS[name].values())
    refs+=list(map(Path,g.cache_file.unique()))
    fingerprint=hashlib.sha256(json.dumps(dict(
        version='nb05_rgb_A_10DSM_4MS_common_001',fold=int(fold),mosaic=mosaic,
        selected=g[['cache_file','cache_index','row','col']].astype(str).values.tolist(),
        files=sig(refs)),sort_keys=True).encode()).hexdigest()
    dest=CACHE/f'f{fold}_{hashlib.sha1(mosaic.encode()).hexdigest()[:10]}_{fingerprint[:16]}.npz'
    if dest.is_file():
        with np.load(dest,allow_pickle=False) as z:
            assert str(z['fingerprint'].item())==fingerprint, 'Existing chip cache fingerprint mismatch'
            n=len(z['rgb'])
        print('REUSE joint chips:',fold,mosaic,n,flush=True)
        CACHE_ROWS.append(dict(fold=fold,mosaic=mosaic,path=str(dest),fingerprint=fingerprint,n=n))
        continue
    examples=[]
    with ExitStack() as stack:
        rgb=stack.enter_context(rasterio.open(MOSAICS[mosaic]))
        dsm=stack.enter_context(rasterio.open(DSM_BY_MOSAIC[mosaic]))
        vrts=mosaic_ms_vrts(rgb,stack,mosaic)
        # Group by source NPZ to avoid repeated Drive decompression.
        for path,records in g.groupby('cache_file',sort=True):
            with np.load(path,allow_pickle=False) as z:
                for q in records.itertuples(index=False):
                    j=int(q.cache_index)
                    win=Window(int(q.col),int(q.row),256,256)
                    try:
                        features,dsm_valid=feature_tile(rgb,dsm,win)
                        ms,ms_valid,flight=choose_complete_flight(vrts,win)
                        y=np.asarray(z['yA'][j],dtype=np.uint8)
                        weight=np.asarray(z['wA'][j],dtype=bool)&dsm_valid&ms_valid
                        if int(weight.sum())<MIN_SUPERVISED_PIXELS:continue
                        examples.append(dict(rgb=z['rgb'][j].astype('uint8'),
                            dsm=features[3:].astype('float16'),
                            ms=ms.astype('float16'),y=y,w=weight.astype('uint8'),
                            site=str(q.site),role=inner_role(q,rgb),flight=str(flight)))
                    except Exception as err:
                        raise RuntimeError(f'Chip failure {fold} {mosaic} {q.row} {q.col}') from err
    assert examples,('No jointly valid chips',fold,mosaic)
    arrays={
        'rgb':np.stack([q['rgb'] for q in examples]),
        'dsm':np.stack([q['dsm'] for q in examples]),
        'ms':np.stack([q['ms'] for q in examples]),
        'y':np.stack([q['y'] for q in examples]),
        'w':np.stack([q['w'] for q in examples]),
        'site':np.array([q['site'] for q in examples]),
        'role':np.array([q['role'] for q in examples]),
        'flight':np.array([q['flight'] for q in examples]),
        'fingerprint':np.array(fingerprint)
    }
    tmp=dest.with_name(dest.name+'.tmp.npz')
    np.savez_compressed(tmp,**arrays);os.replace(tmp,dest)
    CACHE_ROWS.append(dict(fold=fold,mosaic=mosaic,path=str(dest),
                           fingerprint=fingerprint,n=len(examples)))
    print('SAVED joint chips:',fold,mosaic,len(examples),flush=True)
    del examples,arrays;gc.collect()
CACHES=pd.DataFrame(CACHE_ROWS)
CACHES.to_csv(OUT/'03_joint_chip_cache_manifest.csv',index=False)
assert set(CACHES.fold)=={0,1,2}
print('Cached common-support training chips total:',CACHES.n.sum())
display(CACHES)


## 04 — Train RGB+DSM and RGB+MS models, identical training support; checkpoints resumable

Both networks start from the original fold-specific RGB Model A weights. The extra input weights initialize at zero; initial predictions therefore equal RGB Model A before fine-tuning. Select epoch using only the inner training/validation split. **This is not a refit of RGB on identical support**, so report the training asymmetry explicitly.

In [ ]:
import torch,torch.nn as nn,torch.nn.functional as F
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader,Dataset
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE=='cuda','Select Colab GPU for training and inference'
MODEL_DIR=OUT/'04_fusion_models'
MODEL_DIR.mkdir(exist_ok=True)
EPOCHS=12
STEPS=90
BATCH=6
LR=8e-5
SEED=20261119

def model_A(fold):
    path=NB03/f'fold{fold}_A_{TRAIN["training_key"][:18]}.pt'
    ck=torch.load(path,map_location='cpu',weights_only=False)
    assert ck['training_key']==TRAIN['training_key'] and ck['arm']=='A' and ck['fold']==fold
    net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1)
    net.load_state_dict({k.removeprefix('net.'):v for k,v in ck['model'].items()},strict=True)
    return net

def fusion_model(fold,mode):
    extra=10 if mode=='DSM' else 4
    net=smp.Unet(encoder_name='resnet18',encoder_weights=None,
                 in_channels=3+extra,classes=1)
    original=model_A(fold).state_dict()
    state=net.state_dict()
    for k,v in state.items():
        if k=='encoder.conv1.weight':
            assert v.shape[1]==3+extra and original[k].shape[1]==3
            v[:,:3]=original[k]
            v[:,3:]=0
        else:
            assert k in original and v.shape==original[k].shape,k
            v.copy_(original[k])
    net.load_state_dict(state,strict=True)
    return net

def read_joint_fold(fold):
    rows=[]
    for rec in CACHES[CACHES.fold.eq(fold)].itertuples():
        with np.load(rec.path,allow_pickle=False) as z:
            assert str(z['fingerprint'].item())==rec.fingerprint
            n=len(z['y'])
            for j in range(n):
                rows.append({k:np.array(z[k][j]) for k in ['rgb','dsm','ms','y','w']}
                            |{'site':str(z['site'][j]),'role':str(z['role'][j])})
    tr=[q for q in rows if q['role']=='train']
    va=[q for q in rows if q['role']=='val']
    assert len(tr)>=30 and len(va)>=5,(fold,len(tr),len(va))
    return tr,va

def dsm_norm(tr):
    ss=np.zeros(10,np.float64);sq=np.zeros(10,np.float64);n=0
    for q in tr:
        valid=q['w'][::8,::8].astype(bool)
        if not valid.any():continue
        vals=q['dsm'].astype('float32')[:,::8,::8][:,valid]
        ss+=vals.sum(axis=1);sq+=(vals.astype('float64')**2).sum(axis=1);n+=vals.shape[1]
    assert n>1000
    mu=ss/n
    sd=np.sqrt(np.maximum(sq/n-mu*mu,1e-4))
    return mu.astype('float32'),sd.astype('float32')

class JointDataset(Dataset):
    def __init__(self,rows,mode,mu,sd,augment):
        self.rows=rows;self.mode=mode;self.mu=mu[:,None,None];self.sd=sd[:,None,None];self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        z=self.rows[i]
        rgb=z['rgb'].astype('float32').transpose(2,0,1)/255.
        aux=(np.clip((z['dsm'].astype('float32')-self.mu)/self.sd,-6,6)
             if self.mode=='DSM' else z['ms'].astype('float32'))
        x=np.concatenate([rgb,aux],axis=0)
        y=z['y'].astype('float32');w=z['w'].astype('float32')
        if self.augment:
            k=int(torch.randint(4,(1,)).item())
            x=np.rot90(x,k,axes=(1,2));y=np.rot90(y,k);w=np.rot90(w,k)
            if torch.rand(())<.5:
                x=x[:,:,::-1];y=y[:,::-1];w=w[:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(w)))
def masked_bce(logits,y,w):
    l=F.binary_cross_entropy_with_logits(logits.float(),y,reduction='none')
    return (l*w).sum()/w.sum().clamp_min(1.)

@torch.no_grad()
def validate(net,loader):
    net.eval();loss_sum=0.;count=0
    for x,y,w in loader:
        x,y,w=x.to(DEVICE),y.to(DEVICE),w.to(DEVICE)
        with torch.autocast('cuda',enabled=DEVICE=='cuda'):
            logits=net(x)[:,0]
        v=F.binary_cross_entropy_with_logits(logits.float(),y,reduction='none')
        loss_sum+=float((v*w).sum().item());count+=int(w.sum().item())
    return loss_sum/max(count,1)

def fit(fold,mode,tr,va,mu,sd):
    dest=MODEL_DIR/f'fold{fold}_RGB_{mode}_v1.pt'
    fingerprint=hashlib.sha256(json.dumps(dict(
        base_train_key=TRAIN['training_key'],split=SPLIT_CONTRACT['csv_sha256'],
        mode=mode,fold=fold,epochs=EPOCHS,steps=STEPS,batch=BATCH,lr=LR,
        cache_fingerprints=sorted(CACHES[CACHES.fold.eq(fold)].fingerprint.tolist()),
        seed=SEED),sort_keys=True).encode()).hexdigest()
    if dest.exists():
        ck=torch.load(dest,map_location='cpu',weights_only=False)
        assert ck['fingerprint']==fingerprint and ck['complete']
        print('REUSE trained:',mode,fold);return
    torch.manual_seed(SEED+fold)
    net=fusion_model(fold,mode).to(DEVICE)
    opt=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=1e-4)
    train_ds=JointDataset(tr,mode,mu,sd,True)
    val_dl=DataLoader(JointDataset(va,mode,mu,sd,False),batch_size=BATCH,shuffle=False,num_workers=0)
    rng=np.random.default_rng(SEED+fold)
    best=float('inf');best_state=None;best_epoch=0;hist=[]
    for epoch in range(EPOCHS):
        net.train();losses=[];t0=time.perf_counter()
        for step in range(STEPS):
            indexes=rng.integers(0,len(train_ds),size=BATCH)
            batch=[train_ds[int(k)] for k in indexes]
            x,y,w=[torch.stack([q[k] for q in batch]).to(DEVICE) for k in range(3)]
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=DEVICE=='cuda'):
                logits=net(x)[:,0]
                loss=masked_bce(logits,y,w)
            loss.backward();opt.step()
            losses.append(float(loss.detach().item()))
        val=validate(net,val_dl)
        hist.append(dict(fold=fold,mode=mode,epoch=epoch+1,
                         train_loss=np.mean(losses),inner_val_BCE=val,
                         seconds=time.perf_counter()-t0))
        if val<best:
            best=val;best_epoch=epoch+1
            best_state={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}
        print('TRAIN',fold,mode,epoch+1,'/',EPOCHS,
              'train',round(np.mean(losses),4),'val',round(val,4),flush=True)
    tmp=dest.with_name(dest.name+'.tmp')
    torch.save(dict(fingerprint=fingerprint,complete=True,fold=fold,mode=mode,
                    baseline_training_key=TRAIN['training_key'],state=best_state,
                    norm_mean=mu,norm_std=sd,best_epoch=best_epoch,
                    best_val_BCE=best,history=hist),tmp)
    os.replace(tmp,dest)
    pd.DataFrame(hist).to_csv(MODEL_DIR/f'fold{fold}_{mode}_history.csv',index=False)
    del net,opt;gc.collect();torch.cuda.empty_cache()

for fold in range(3):
    tr,va=read_joint_fold(fold)
    mu,sd=dsm_norm(tr)
    print('FOLD',fold,'train',len(tr),'inner validation',len(va))
    for mode in ('DSM','MS'):
        fit(fold,mode,tr,va,mu,sd)
    del tr,va;gc.collect();torch.cuda.empty_cache()
print('Training complete: two fusion arms x three spatial folds')


## 05 — Exact same held-out unit masks, matched-pixel inference, resumable per unit

Inference uses NB04's 256-pixel tiles, 128-pixel overlap and positive Hann weights. It predicts frozen RGB, RGB+DSM and RGB+MS together; validates **only common RGB/DSM/MS-valid reference pixels**. The expensive DSM/MS tile reads are shared between both fusion models. Units without complete MS support are **recorded as unavailable**, not assigned zero cover.

In [ ]:

from scipy.ndimage import binary_dilation
from rasterio.windows import transform as window_transform
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import box
from pyproj import Transformer
from shapely.ops import transform as shape_transform
TILE=256
STRIDE=128
BATCH_INFER=6
PROB_THRESHOLD=.5
def to_ds_geom(g,ds):
    if ds.crs==rasterio.crs.CRS.from_user_input(REF_CRS):return g
    tx=Transformer.from_crs(REF_CRS,ds.crs,always_xy=True)
    return shape_transform(tx.transform,g)
def burn(g,ds,win):
    shape=(int(win.height),int(win.width))
    if g is None or g.is_empty:return np.zeros(shape,bool)
    bbox=box(*rasterio.windows.bounds(win,ds.transform))
    if not g.intersects(bbox):return np.zeros(shape,bool)
    clipped=g.intersection(bbox)
    return rasterize([(clipped,1)],out_shape=shape,
                     transform=window_transform(win,ds.transform),fill=0,dtype='uint8').astype(bool)
def rgb_read(ds,win):
    data=ds.read([1,2,3],window=win,masked=True)
    valid=~np.ma.getmaskarray(data).any(axis=0)
    valid &= np.isfinite(np.asarray(data.data,dtype=np.float32)).all(axis=0)
    if ds.count>=4:
        alpha=ds.read(4,window=win,masked=True)
        valid &= ~np.ma.getmaskarray(alpha) & (np.asarray(alpha.filled(0))>0)
    x=np.clip(np.asarray(data.filled(0)),0,255).astype('uint8').transpose(1,2,0)
    return x,valid


def origins(length,tile=TILE,stride=STRIDE):
    if length<=tile:return [0]
    vals=list(range(0,length-tile+1,stride))
    if vals[-1]!=length-tile:vals.append(length-tile)
    return vals
hann=np.hanning(258)[1:-1]
HANN=np.outer(hann,hann).astype('float32')
HANN/=HANN.max()

def get_models(fold):
    models={'RGB':model_A(fold).to(DEVICE).eval()}
    mu=None;sd=None;checkpoints=[]
    for mode in ('DSM','MS'):
        f=MODEL_DIR/f'fold{fold}_RGB_{mode}_v1.pt'
        ck=torch.load(f,map_location='cpu',weights_only=False)
        assert ck['complete'] and ck['baseline_training_key']==TRAIN['training_key']
        m=fusion_model(fold,mode).to(DEVICE)
        m.load_state_dict(ck['state'],strict=True)
        models[mode]=m.eval()
        checkpoints.append((str(f),ck['fingerprint']))
        if mu is None:mu,sd=np.array(ck['norm_mean']),np.array(ck['norm_std'])
        else:
            assert np.allclose(mu,ck['norm_mean']) and np.allclose(sd,ck['norm_std'])
    return models,mu,sd,checkpoints

@torch.inference_mode()
def infer_joint(ds,dsm,vrts,unit_geom,models,mu,sd):
    geom=to_ds_geom(unit_geom,ds)
    bounds=rasterio.windows.from_bounds(*geom.bounds,transform=ds.transform)
    row0=max(0,int(np.floor(bounds.row_off))-TILE)
    col0=max(0,int(np.floor(bounds.col_off))-TILE)
    row1=min(ds.height,int(np.ceil(bounds.row_off+bounds.height))+TILE)
    col1=min(ds.width,int(np.ceil(bounds.col_off+bounds.width))+TILE)
    h=row1-row0;ww=col1-col0
    assert h>0 and ww>0 and h*ww<75_000_000,(h,ww)
    sums={mode:np.zeros((h,ww),np.float32) for mode in ('RGB','DSM','MS')}
    weights=np.zeros((h,ww),np.float32)
    # Tracks agreement that every contributing tile has all auxiliary data valid.
    valid_weight=np.zeros((h,ww),np.float32)
    batch=[]
    def flush():
        if not batch:return
        inp={
            'RGB':torch.from_numpy(np.stack([q['rgb'] for q in batch])).float().to(DEVICE)/255.,
            'DSM':torch.from_numpy(np.stack([q['dsm'] for q in batch])).float().to(DEVICE),
            'MS':torch.from_numpy(np.stack([q['ms'] for q in batch])).float().to(DEVICE)
        }
        for mode,net in models.items():
            with torch.autocast('cuda',enabled=DEVICE=='cuda'):
                probs=torch.sigmoid(net(inp[mode])[:,0]).float().cpu().numpy()
            for rec,p in zip(batch,probs):
                r,c,hh,wwi=rec['loc']
                sums[mode][r:r+hh,c:c+wwi]+=p[:hh,:wwi]*HANN[:hh,:wwi]
        for rec in batch:
            r,c,hh,wwi=rec['loc']
            kern=HANN[:hh,:wwi]
            weights[r:r+hh,c:c+wwi]+=kern
            valid_weight[r:r+hh,c:c+wwi]+=kern*rec['joint'][:hh,:wwi]
        batch.clear()

    for r in origins(h):
        for c in origins(ww):
            hh=min(TILE,h-r);wi=min(TILE,ww-c)
            win=Window(col0+c,row0+r,wi,hh)
            rgb_data,rgb_ok=rgb_read(ds,win)
            dsm13,dsm_ok=feature_tile(ds,dsm,win)
            ms4,ms_ok,flight=choose_complete_flight(vrts,win)
            joint=rgb_ok&dsm_ok&ms_ok
            nr=np.zeros((3,TILE,TILE),np.uint8)
            nd=np.zeros((13,TILE,TILE),np.float32)
            nm=np.zeros((7,TILE,TILE),np.float32)
            nr[:,:hh,:wi]=rgb_data.transpose(2,0,1)
            nd[:3,:hh,:wi]=nr[:,:hh,:wi].astype('float32')/255.
            nd[3:,:hh,:wi]=np.clip(
                (dsm13[3:]-mu[:,None,None])/sd[:,None,None],-6,6)
            nm[:3,:hh,:wi]=nd[:3,:hh,:wi]
            nm[3:,:hh,:wi]=ms4
            batch.append(dict(rgb=nr,dsm=nd,ms=nm,loc=(r,c,hh,wi),joint=joint))
            if len(batch)>=BATCH_INFER:flush()
    flush()
    assert weights.min()>0
    preds={mode:np.clip(sums[mode]/weights,0,1) for mode in sums}
    common=(valid_weight>=weights-1e-5)
    win=Window(col0,row0,ww,h)
    x,rv=rgb_read(ds,win)
    footprint=burn(geom,ds,win)
    roi=burn(to_ds_geom(ROI_GEOM,ds),ds,win)
    common&=rv&footprint&roi
    return preds,common,win

from sklearn.metrics import roc_auc_score,average_precision_score
HIST_BINS=np.linspace(0,1,101)
def truth_mask(unit_type,ds,win,unit_mask):
    # Positives come only from the selected exhaustive reference source.
    positive=KVO_POS if unit_type=='circle' else NADINE_POS
    pos=burn(to_ds_geom(positive,ds),ds,win)&unit_mask
    conflict=burn(to_ds_geom(CONFLICT,ds),ds,win)
    supervised=unit_mask&~conflict
    # No background labels inferred from non-exhaustive circles.
    if unit_type=='plot':
        # Explicit negative crowns are contradictory with positive crowns when overlapping.
        positive_negative=burn(to_ds_geom(NADINE_NEG,ds),ds,win)&pos
        supervised&=~positive_negative
    return pos,supervised

def scores(y,p):
    y=np.asarray(y,dtype=bool);p=np.asarray(p,dtype=np.float32)
    q=p>=PROB_THRESHOLD;tp=int((q&y).sum());fp=int((q&~y).sum());fn=int((~q&y).sum());tn=int((~q&~y).sum())
    f1=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else np.nan
    iou=tp/(tp+fp+fn) if (tp+fp+fn) else np.nan
    precision=tp/(tp+fp) if tp+fp else np.nan
    recall=tp/(tp+fn) if tp+fn else np.nan
    obs=100*y.mean();pred=100*p.mean()
    out=dict(n_pixels=len(y),n_pos=int(y.sum()),n_neg=int((~y).sum()),
             observed_cover_pp=obs,predicted_cover_pp=pred,cover_error_pp=pred-obs,
             abs_cover_error_pp=abs(pred-obs),brier=float(np.mean((p-y)**2)),
             tp=tp,fp=fp,fn=fn,tn=tn,F1=f1,IoU=iou,precision=precision,recall=recall)
    if y.any() and (~y).any():
        out['roc_auc']=float(roc_auc_score(y,p));out['average_precision']=float(average_precision_score(y,p))
    else:out.update(roc_auc=np.nan,average_precision=np.nan)
    hist_pos=np.histogram(p[y],bins=HIST_BINS)[0]
    hist_neg=np.histogram(p[~y],bins=HIST_BINS)[0]
    return out,hist_pos,hist_neg


def json_default(obj):
    if isinstance(obj,np.generic):return obj.item()
    if isinstance(obj,np.ndarray):return obj.tolist()
    raise TypeError(type(obj).__name__)

RESULT_DIR=OUT/'05_per_unit'
RESULT_DIR.mkdir(exist_ok=True)
summary=[]
for fold in range(3):
    models,mu,sd,ckpts=get_models(fold)
    ck_hash=hashlib.sha256(json.dumps(ckpts,sort_keys=True).encode()).hexdigest()
    for z in EVAL[EVAL.fold.eq(fold)].itertuples(index=False):
        fname=RESULT_DIR/f'f{fold}_{z.unit_type}_{str(z.unit_id).replace("/","_")}.npz'
        if fname.is_file():
            with np.load(fname,allow_pickle=False) as obj:
                assert str(obj['checkpoint_hash'].item())==ck_hash
                metadata=json.loads(str(obj['metadata'].item()))
                assert metadata['unit_id']==str(z.unit_id)
            print('REUSE',z.site,z.unit_type,z.unit_id,flush=True)
            continue
        with ExitStack() as stack:
            ds=stack.enter_context(rasterio.open(MOSAICS[z.mosaic]))
            dsm=stack.enter_context(rasterio.open(DSM_BY_MOSAIC[z.mosaic]))
            vrts=mosaic_ms_vrts(ds,stack,z.mosaic)
            preds,common,win=infer_joint(ds,dsm,vrts,
                REF_UNITS[(str(z.unit_type),str(z.unit_id))],models,mu,sd)
            truth,supervised=truth_mask(str(z.unit_type),ds,win,common)
            n=int(supervised.sum())
            report={}
            if n>0:
                y=truth[supervised].astype('uint8')
                for mode in ('RGB','DSM','MS'):
                    score,*_=scores(y,preds[mode][supervised])
                    report[mode]=score
            metadata=dict(fold=int(z.fold),site=str(z.site),
                unit_type=str(z.unit_type),unit_id=str(z.unit_id),
                spatial_group=str(z.spatial_group),mosaic=str(z.mosaic),
                n_common=n,missing_common_data=n==0)
        record=dict(checkpoint_hash=np.array(ck_hash),
                    metadata=np.array(json.dumps(metadata)),
                    results=np.array(json.dumps(report,default=json_default)))
        tmp=fname.with_name(fname.name+'.tmp.npz')
        np.savez_compressed(tmp,**record);os.replace(tmp,fname)
        print('COMPLETED',z.site,z.unit_type,z.unit_id,'common pixels',n,flush=True)
    del models;gc.collect();torch.cuda.empty_cache()
print('Inference cached for 106 original held-out reference units.')


## 06 — NB04-consistent combined circle+plot evaluation, figures and paired uncertainties

Primary comparison = **equal-footprint** metrics on the same exact common-valid pixels; supplementary valid-pixel weighting. Also report site breakdown, strata, all 106 support counts and pixel-pooled confusion. All 4 difficult MOS references are retained when complete spectral support exists.

In [ ]:

from sklearn.metrics import r2_score
from IPython.display import display
import matplotlib.pyplot as plt
rows=[];support=[]
for z in EVAL.itertuples(index=False):
    p=RESULT_DIR/f'f{int(z.fold)}_{z.unit_type}_{str(z.unit_id).replace("/","_")}.npz'
    assert p.is_file(),p
    with np.load(p,allow_pickle=False) as x:
        meta=json.loads(str(x['metadata'].item()))
        res=json.loads(str(x['results'].item()))
    support.append(meta)
    for mode,q in res.items():
        rows.append({**meta,'model':'RGB' if mode=='RGB' else 'RGB+'+mode,**q})
S=pd.DataFrame(support);RESULTS=pd.DataFrame(rows)
assert len(S)==106 and not S.duplicated(['unit_type','unit_id']).any()
S.to_csv(OUT/'06_all_106_unit_support.csv',index=False)
assert S.n_common.gt(0).any(),'No complete RGB+DSM+MS support; diagnose source registration'
ncommon=int(S.n_common.gt(0).sum())
assert len(RESULTS)==ncommon*3
for k,g in RESULTS.groupby(['unit_type','unit_id']):
    assert g.model.nunique()==3
    assert g.n_pixels.nunique()==1
    assert g.observed_cover_pp.max()-g.observed_cover_pp.min()<1e-9
RESULTS.to_csv(OUT/'06_all_three_models_matched_unit_metrics.csv',index=False)
print('FULL NB04 UNITS',len(BASELINE),'| THREE-WAY MATCHED UNITS',ncommon,'| ZERO COMMON SUPPORT',106-ncommon)
display(S.groupby(['site','unit_type']).agg(
    total=('n_common','size'),matched=('n_common',lambda v:int((v>0).sum())),
    common_pixels=('n_common','sum')))
print('Not all 106 units have the same source coverage; missing units are NOT zero predictions.')

# Copy NB04 exact weighted metrics definitions, independently of notebook globals
def wm(x,w):
    x=np.asarray(x,float);w=np.asarray(w,float)
    valid=np.isfinite(x)&np.isfinite(w)&(w>0)
    return float(np.average(x[valid],weights=w[valid])) if valid.any() else np.nan

def weighted_quantile(x,w,qs):
    x=np.asarray(x,float);w=np.asarray(w,float)
    valid=np.isfinite(x)&np.isfinite(w)&(w>0)
    x=x[valid];w=w[valid]
    if not len(x):return np.full(len(qs),np.nan)
    ix=np.argsort(x);x=x[ix];w=w[ix]
    c=(np.cumsum(w)-w/2)/w.sum()
    return np.interp(qs,c,x)

def weights(g,method):
    if method=='Equal units':return np.ones(len(g),float)
    if method=='Balanced circle/plot':
        n=g.unit_type.value_counts()
        return g.unit_type.map(lambda t:1/n[t]).to_numpy(float)
    if method=='Valid-pixel weighted':return g.n_pixels.to_numpy(float)
    raise ValueError(method)

def summarize(g,method):
    w=weights(g,method)
    obs=g.observed_cover_pp.to_numpy(float)
    pred=g.predicted_cover_pp.to_numpy(float)
    error=pred-obs
    mae=wm(np.abs(error),w)
    rmse=np.sqrt(wm(error**2,w))
    bias=wm(error,w)
    var=wm((obs-wm(obs,w))**2,w)
    sd=np.sqrt(var)
    q1,q3=weighted_quantile(obs,w,[.25,.75])
    # Weighted means of unit-level segmentation metrics (not pixel-pooled F1).
    result={'n':len(g),'F1':wm(g.F1,w),'IoU':wm(g.IoU,w),
            'Precision':wm(g.precision,w),'Recall':wm(g.recall,w),
            'R2':1-rmse**2/var if var>0 else np.nan,
            'MAE_pp':mae,'RMSE_pp':rmse,'Bias_pp':bias,
            'RPD':sd/rmse if rmse>0 else np.nan,
            'RPIQ':(q3-q1)/rmse if rmse>0 else np.nan,
            'Brier':wm(g.brier,w)}
    return result


SCHEMES=['Equal units','Valid-pixel weighted']
REPORT=[]
for (group,gg) in [('ALL',RESULTS),*list(RESULTS.groupby('site'))]:
    for model,g in gg.groupby('model'):
        for method in SCHEMES:
            REPORT.append(dict(Group=group,model=model,Weighting=method,
                               **summarize(g,method)))
REPORT=pd.DataFrame(REPORT)
REPORT.to_csv(OUT/'06_weighted_model_comparison.csv',index=False)
display(REPORT.query('Group == "ALL"').round(3))
print('PRIMARY — equal weight per reference footprint, per site:')
display(REPORT.query('Weighting == "Equal units"').round(3))

# Explicit paired improvement on exact same common-valid reference units.
P=RESULTS.pivot(index=['site','unit_type','unit_id','fold','spatial_group'],
                columns='model',
                values=['observed_cover_pp','predicted_cover_pp','F1','IoU','n_pixels'])
P.columns=['%s_%s'%(a,b) for a,b in P.columns]
P=P.reset_index()
p0=P['observed_cover_pp_RGB'].to_numpy(float)
for mode in ('RGB+DSM','RGB+MS'):
    P['Delta_abs_cover_error_'+mode]=(
        np.abs(P['predicted_cover_pp_'+mode]-p0)
        -np.abs(P['predicted_cover_pp_RGB']-p0))
P.to_csv(OUT/'06_paired_unit_differences.csv',index=False)

# Group bootstrap keeps spatial group dependence; does not treat pixels as independent.
rng=np.random.default_rng(20261119)
boot=[]
for mode in ('RGB+DSM','RGB+MS'):
    g=P.copy()
    keys=g.spatial_group.unique()
    vals=[]
    for _ in range(1000):
        draw=rng.choice(keys,size=len(keys),replace=True)
        sampled=pd.concat([g[g.spatial_group.eq(t)] for t in draw],ignore_index=True)
        w=weights(sampled,'Equal units')
        vals.append(wm(sampled['Delta_abs_cover_error_'+mode],w))
    point=wm(g['Delta_abs_cover_error_'+mode],
             weights(g,'Equal units'))
    boot.append(dict(model=mode,delta_MAE_pp=point,
                     CI95_low=np.percentile(vals,2.5),
                     CI95_high=np.percentile(vals,97.5),n_units=len(g),
                     n_groups=len(keys)))
BOOT=pd.DataFrame(boot)
BOOT.to_csv(OUT/'06_spatial_group_bootstrap.csv',index=False)
print('Paired delta absolute cover error (negative = improvement)')
display(BOOT.round(3))

colors={'circle':'#1675B5','plot':'#DC8140'}
fig,axes=plt.subplots(1,3,figsize=(17,5),constrained_layout=True)
for ax,mode in zip(axes,('RGB','RGB+DSM','RGB+MS')):
    data=RESULTS[RESULTS.model.eq(mode)]
    for typ,part in data.groupby('unit_type'):
        ax.scatter(part.observed_cover_pp,part.predicted_cover_pp,
                   s=35,alpha=.67,c=colors[typ],
                   marker='o' if typ=='circle' else 's',
                   label=f'{typ} (n={len(part)})')
    ax.plot([0,100],[0,100],'k--',lw=1)
    ax.set(xlim=(-2,102),ylim=(-2,102),title=mode,
           xlabel='Reference Musanga cover (%)',
           ylabel='Predicted Musanga cover (%)')
    ax.set_aspect('equal',adjustable='box');ax.grid(alpha=.15);ax.legend(frameon=False)
fig.suptitle('Held-out full-unit predictions | exact common RGB–DSM–MS pixels')
fig.savefig(OUT/'06_three_arm_matched_scatter.png',dpi=210,bbox_inches='tight')
plt.show()

# Forest-specific scatterplots of three modalities.
fig,axs=plt.subplots(4,3,figsize=(14,17),constrained_layout=True)
sites=sorted(RESULTS.site.unique())
for i,site in enumerate(sites):
    for j,mode in enumerate(('RGB','RGB+DSM','RGB+MS')):
        ax=axs[i,j]
        d=RESULTS[(RESULTS.site.eq(site))&(RESULTS.model.eq(mode))]
        for typ,part in d.groupby('unit_type'):
            ax.scatter(part.observed_cover_pp,part.predicted_cover_pp,
                       color=colors[typ],alpha=.7,s=26,
                       marker='o' if typ=='circle' else 's')
        ax.plot([0,100],[0,100],'k--',lw=1)
        ax.set(xlim=(-2,102),ylim=(-2,102),
               title=f'{site} | {mode} | n={len(d)}',
               xlabel='Reference cover (%)',ylabel='Predicted cover (%)')
        ax.grid(alpha=.12)
fig.savefig(OUT/'06_three_arm_scatter_by_forest.png',dpi=180,bbox_inches='tight')
plt.show()

# Cover strata for high-cover MOS failure interpretation.
bins=[-.001,1,5,20,40,100.001]
labels=['0–1%','1–5%','5–20%','20–40%','>40%']
RESULTS['cover_stratum']=pd.cut(RESULTS.observed_cover_pp,bins=bins,
                               labels=labels,include_lowest=True)
COVER=[]
for (site,model,stratum),g in RESULTS.groupby(
        ['site','model','cover_stratum'],observed=True):
    COVER.append(dict(site=site,model=model,cover_stratum=str(stratum),
                      **summarize(g,'Equal units')))
COVER=pd.DataFrame(COVER)
COVER.to_csv(OUT/'06_cover_strata_by_site.csv',index=False)
print('MOS high-cover >40%, including problematic units if they have complete MS support:')
display(COVER[(COVER.site=='MOS')&(COVER.cover_stratum=='>40%')].round(3))

# Separate NB04 106-unit *full RGB* baseline from the matched-support RGB scores.
nb04=BASELINE.copy()
nb04_summary={**summarize(nb04,'Equal units'),
              'model':'NB04 RGB Model A','Group':'ALL full RGB only','n':len(nb04)}
matched=REPORT[(REPORT.Group=='ALL')&(REPORT.Weighting=='Equal units')].copy()
display(pd.concat([pd.DataFrame([nb04_summary]),matched],ignore_index=True).round(3))
manifest=dict(schema='NB05_matched_v1',baseline_training_key=TRAIN['training_key'],
              original_nb04_units=106,matched_units=ncommon,
              arms=['RGB','RGB+DSM10','RGB+MS4'],
              note='Full NB04 RGB baseline is not on identical pixels as matched-sensor comparison.',
              exclusions=[],fusion_training_asymmetry='RGB frozen, DSM/MS initialized and fine-tuned',primary_weighting='equal reference footprints as NB04 v4_007')
(OUT/'06_reporting_contract.json').write_text(json.dumps(manifest,indent=2))
print('COMPLETE NB05 report:',OUT)


## Interpretation and limitations

- **Primary:** paired equal-footprint comparison on identical common-valid pixels. Retain all original 106 unit identities and explicit counts of unavailable MS support. No imputation, no selective removal of the four MOS cases.
- **Secondary:** site-specific R² / MAE / RMSE / bias, F1 / IoU, equal-footprint primary and valid-pixel supplementary weightings, >40% MOS, RPD / RPIQ, spatial-group bootstrap and scatterplots.
- **Original NB04 baseline:** 106-unit RGB results remain authoritative for the full RGB evaluation. The NB05 RGB row is **recomputed on matched sensor support**, so different metrics are expected.
- **Not a perfectly symmetric ablation:** RGB checkpoint frozen from NB03, while auxiliary-modality models receive additional fine-tuning. If the fusion models appear better, a trained RGB-only common-support control should be run before claiming an effect attributable specifically to the auxiliary sensor.
- **Registration QA required:** DSM and multispectral rasters are warped to the RGB grid and tested for valid coverage, but this alone cannot certify sub-crown alignment.
- **No claims of improvement until notebook runs.** Large 1m-context DSM windows make extraction expensive; cache and resume are designed to avoid wasting previous work. Historical code is the source for exact multiscale definitions, not for old model scores.

Historical references (in Git history): `03_DSM_Multiscale_Context_v4_009`, `04_MS_Quality_and_Relative_Spectra_v4_019`, `05_Harmonized_RGB_DSM_MS_OptionB_v4_026`. Current baseline references: NB03 v4_002 and NB04 v4_006.